In [39]:
# instance_supcon_loss 快速验证
# 期望：类内可分且两视图一致时损失较低；随机特征或打乱标签时损失明显升高。

import sys
from pathlib import Path

import torch
import torch.nn.functional as F

# 保证能从项目根目录导入 utils.losses
for _root in [Path.cwd(), Path.cwd().parent]:
    if (_root / "utils" / "losses.py").exists():
        sys.path.insert(0, str(_root.resolve()))
        break
else:
    raise RuntimeError("请在 ADMamba 项目根目录或其 utils 目录下打开本 notebook")

from utils.losses import instance_supcon_loss

torch.manual_seed(0)
B, T, C = 256, 128, 18
n_cls = 3
# 三类样本数固定为 46、85、125（合计须等于 B）
counts = [46, 85, 125]
assert sum(counts) == B and len(counts) == n_cls and min(counts) >= 2

# 三个类别在 R^C 上取近似正交方向，类间差异大
centers = torch.zeros(n_cls, C)
for k in range(n_cls):
    # centers[k, k * 6 : k * 6 + 6] = 1.0  # 各占用不重叠维度
    centers[k, : ] = 1.0  # 各占用不重叠维度
# centers = torch.nn.functional.normalize(centers, p=2, dim=-1)

labels = torch.cat([torch.full((counts[k],), k, dtype=torch.long) for k in range(n_cls)])

# 视图 ori：类中心 + 小噪声（沿时间维独立）
noise_scale = 0.0
emb_base = centers[labels]  # [B, C]
embed_ori = emb_base.unsqueeze(1).expand(-1, T, -1).clone()
embed_ori = embed_ori + noise_scale * torch.randn(B, T, C)

# 视图 aug：同类同结构，另一组噪声（与 ori 仍相近 → 正样本合理）
embed_aug = emb_base.unsqueeze(1).expand(-1, T, -1).clone()
embed_aug = embed_aug + noise_scale * torch.randn(B, T, C)

# embed_ori 的交叉熵：时间维池化 [B,T,C] -> [B,C]，再分类；此处用与构造数据相同的 centers 作固定线性头，便于与合成设定一致
h_ori = embed_ori.mean(dim=1)
logits_ori = F.linear(h_ori, centers)
loss_ce_ori = F.cross_entropy(logits_ori, labels)

supcon_temp = 0.1
loss_sep = instance_supcon_loss(embed_ori, embed_aug, labels, supcon_temp=supcon_temp)

# 对照 1：完全随机嵌入 → 对比学习难以对齐，损失应更大（同一 zr 亦用于随机 CE）
zr = torch.randn(B, T, C)
za = torch.randn(B, T, C)
loss_rand = instance_supcon_loss(zr, za, labels, supcon_temp=supcon_temp)
h_rand = zr.mean(dim=1)
logits_rand = F.linear(h_rand, centers)
loss_ce_rand = F.cross_entropy(logits_rand, labels)

# 对照 2：特征仍可分，但标签随机打乱 → 错误正样本，损失应远高于 separable
perm = torch.randperm(B)
wrong_labels = labels[perm]
loss_wrong_label = instance_supcon_loss(embed_ori, embed_aug, wrong_labels, supcon_temp=supcon_temp)

print("=== instance_supcon_loss 形状检查 ===")
print(f"embed_ori: {tuple(embed_ori.shape)}, labels: {tuple(labels.shape)}")

print("\n=== 数值结果（标量）===")
print(f"SupCon 类可分 + 两视图一致: loss = {loss_sep.item():.6f}")
print(f"SupCon 随机特征:            loss = {loss_rand.item():.6f}")
print(f"SupCon 可分但标签打乱:      loss = {loss_wrong_label.item():.6f}")
print(f"CE(embed_ori 池化后):       loss = {loss_ce_ori.item():.6f}")
print(f"CE(同一时间维池化的随机 zr): loss = {loss_ce_rand.item():.6f}")

print("\n=== 合理性判断 ===")
ok = loss_sep.item() < loss_rand.item() and loss_sep.item() < loss_wrong_label.item()
print(f"可分场景损失 < 随机 且 < 打乱标签: {ok}")
if ok:
    print("通过：与监督对比学习目标一致（同类拉近、异类远离在当前设置下成立）。")
else:
    print("未通过：请检查实现或增大类间差异 / 调整温度后再试。")

=== instance_supcon_loss 形状检查 ===
embed_ori: (256, 128, 18), labels: (256,)

=== 数值结果（标量）===
SupCon 类可分 + 两视图一致: loss = 6.236370
SupCon 随机特征:            loss = 8.698006
SupCon 可分但标签打乱:      loss = 6.236370
CE(embed_ori 池化后):       loss = 1.098612
CE(同一时间维池化的随机 zr): loss = 1.098612

=== 合理性判断 ===
可分场景损失 < 随机 且 < 打乱标签: False
未通过：请检查实现或增大类间差异 / 调整温度后再试。


In [2]:
emb_base = centers[[0, 1]]
print("类中心：", emb_base.shape)
print("类中心：", emb_base)
# 需先运行上一单元，得到 embed_ori [B,T,C]、labels [B]
# 同类两条样本：时间维均值池化后再对特征维 L2 归一化，余弦相似度 = 归一化后向量点积
import torch.nn.functional as F
same = (labels == labels[0]).nonzero(as_tuple=True)[0]
i, j = same[0].item(), same[1].item()
v1 = embed_ori[i].mean(dim=0)
v2 = embed_ori[j].mean(dim=0)
v1n = F.normalize(v1, p=2, dim=-1)
v2n = F.normalize(v2, p=2, dim=-1)
cos_sim = (v1n * v2n).sum()
print(f"样本下标 ({i}, {j})，同类标签 y = {labels[i].item()}")
print(f"余弦相似度: {cos_sim.item():.6f}")

类中心： torch.Size([2, 18])
类中心： tensor([[1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.]])
样本下标 (0, 1)，同类标签 y = 0
余弦相似度: 1.000000


In [3]:
# 需先运行首单元，得到 embed_ori [B,T,C]、labels [B]
# 异类两条样本：时间维均值池化 → 特征维 L2 归一化 → 余弦相似度（点积）
import torch.nn.functional as F

i = (labels == 0).nonzero(as_tuple=True)[0][0].item()
j = (labels == 1).nonzero(as_tuple=True)[0][0].item()

v1 = embed_ori[i].mean(dim=0)
v2 = embed_ori[j].mean(dim=0)
v1n = F.normalize(v1, p=2, dim=-1)
v2n = F.normalize(v2, p=2, dim=-1)
cos_sim = (v1n * v2n).sum()

print(f"样本下标 ({i}, {j})，异类标签 y = {labels[i].item()} vs {labels[j].item()}")
print(f"余弦相似度: {cos_sim.item():.6f}")

样本下标 (0, 86)，异类标签 y = 0 vs 1
余弦相似度: 0.000000


In [14]:
z_ori = F.normalize(embed_ori, p=2, dim=-1)
z_aug = F.normalize(embed_aug, p=2, dim=-1)
# [2B, T, C] -> [T, 2B, C]，每个 t 上计算 (2B × 2B) 样本间相似度
features = torch.cat([z_ori, z_aug], dim=0)
z = features.transpose(0, 1).contiguous()
sim_matrix = torch.bmm(z, z.transpose(1, 2)) / supcon_temp  # [T, 2B, 2B]
print("相似度矩阵：", sim_matrix.shape)
# print("相似度矩阵：", sim_matrix)

m0 = sim_matrix[0]
print("sim_matrix[0]：", m0)
# 屏幕上 10.0000 多为打印舍入；float32 里常为 9.999999xx，与 10 严格相等会为 False，须用 isclose
near10 = torch.isclose(m0, m0.new_tensor(10.0), rtol=1e-4, atol=1e-4)
idx_eq10 = near10.nonzero(as_tuple=False)  # [N, 2]，每行为 (行, 列)
n = idx_eq10.shape[0]
print(f"与 10 接近 (isclose) 的元素个数: {n}")
print(f"m0 最大值 / 最小值: {m0.max().item():.8f}, {m0.min().item():.8f}")
if n <= 200:
    print("(行索引, 列索引):\n", idx_eq10)
else:
    print("(行索引, 列索引) 前 50 行:\n", idx_eq10[:50])
    print("...")
    print("(行索引, 列索引) 后 50 行:\n", idx_eq10[-50:])

相似度矩阵： torch.Size([128, 512, 512])
sim_matrix[0]： tensor([[10.0000, 10.0000, 10.0000,  ...,  0.0000,  0.0000,  0.0000],
        [10.0000, 10.0000, 10.0000,  ...,  0.0000,  0.0000,  0.0000],
        [10.0000, 10.0000, 10.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ..., 10.0000, 10.0000, 10.0000],
        [ 0.0000,  0.0000,  0.0000,  ..., 10.0000, 10.0000, 10.0000],
        [ 0.0000,  0.0000,  0.0000,  ..., 10.0000, 10.0000, 10.0000]])
与 10 接近 (isclose) 的元素个数: 87384
m0 最大值 / 最小值: 9.99999905, 0.00000000
(行索引, 列索引) 前 50 行:
 tensor([[ 0,  0],
        [ 0,  1],
        [ 0,  2],
        [ 0,  3],
        [ 0,  4],
        [ 0,  5],
        [ 0,  6],
        [ 0,  7],
        [ 0,  8],
        [ 0,  9],
        [ 0, 10],
        [ 0, 11],
        [ 0, 12],
        [ 0, 13],
        [ 0, 14],
        [ 0, 15],
        [ 0, 16],
        [ 0, 17],
        [ 0, 18],
        [ 0, 19],
        [ 0, 20],
        [ 0, 21],
        [ 0, 22],
        [ 0, 2

In [33]:
labels_full = torch.cat([labels, labels], dim=0)
print("labels_full: ", labels_full)
mask = (labels_full.unsqueeze(0) == labels_full.unsqueeze(1))
print("mask: ", mask)
logits_mask = torch.ones(2 * B, 2 * B)
logits_mask.fill_diagonal_(0.0)
print("logits_mask: ", logits_mask)
mask = mask * logits_mask
print("mask: ", mask)

pos_count = mask.sum(dim=1)
anchor_valid = pos_count > 0

labels_full:  tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 

In [34]:
logits_max, _ = torch.max(sim_matrix, dim=2, keepdim=True)
print("logits_max: ", logits_max)
logits = sim_matrix - logits_max.detach()
print("logits: ", logits)
exp_logits = torch.exp(logits)
print("exp_logits: ", exp_logits)
logits_mask_t = logits_mask.unsqueeze(0).expand(T, -1, -1)
print("logits_mask_t: ", logits_mask_t)
log_prob = logits - torch.log((exp_logits * logits_mask_t).sum(dim=2, keepdim=True) + 1e-6)
print("log_prob: ", log_prob)

logits_max:  tensor([[[10.0000],
         [10.0000],
         [10.0000],
         ...,
         [10.0000],
         [10.0000],
         [10.0000]],

        [[10.0000],
         [10.0000],
         [10.0000],
         ...,
         [10.0000],
         [10.0000],
         [10.0000]],

        [[10.0000],
         [10.0000],
         [10.0000],
         ...,
         [10.0000],
         [10.0000],
         [10.0000]],

        ...,

        [[10.0000],
         [10.0000],
         [10.0000],
         ...,
         [10.0000],
         [10.0000],
         [10.0000]],

        [[10.0000],
         [10.0000],
         [10.0000],
         ...,
         [10.0000],
         [10.0000],
         [10.0000]],

        [[10.0000],
         [10.0000],
         [10.0000],
         ...,
         [10.0000],
         [10.0000],
         [10.0000]]])
logits:  tensor([[[  0.0000,   0.0000,   0.0000,  ..., -10.0000, -10.0000, -10.0000],
         [  0.0000,   0.0000,   0.0000,  ..., -10.0000, -10.0000, -10.0

In [35]:
mean_log_prob_pos = (mask.unsqueeze(0) * log_prob).sum(dim=2) / (pos_count.unsqueeze(0) + 1e-6)
print("mean_log_prob_pos: ", mean_log_prob_pos)
loss = -mean_log_prob_pos[:, anchor_valid].mean(dim=1).mean()
print("loss: ", loss)

mean_log_prob_pos:  tensor([[-5.1418, -5.1418, -5.1418,  ..., -5.1300, -5.1300, -5.1300],
        [-5.1418, -5.1418, -5.1418,  ..., -5.1300, -5.1300, -5.1300],
        [-5.1418, -5.1418, -5.1418,  ..., -5.1300, -5.1300, -5.1300],
        ...,
        [-5.1418, -5.1418, -5.1418,  ..., -5.1300, -5.1300, -5.1300],
        [-5.1418, -5.1418, -5.1418,  ..., -5.1300, -5.1300, -5.1300],
        [-5.1418, -5.1418, -5.1418,  ..., -5.1300, -5.1300, -5.1300]])
loss:  tensor(5.1339)
